<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/813_CBOv2_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nice — this test suite is already doing the most important thing: **locking in your trust guarantees** (determinism, schema guardrails, and executive report shape) without needing LangGraph.

Below is a review focused on (1) correctness, (2) future brittleness, and (3) a few high-leverage tests to add.

---

## What’s strong (keep)

### ✅ Clear separation: loader vs analysis vs report helpers

You’re testing each layer directly, which is exactly right for an orchestrator template.

### ✅ You’re testing “governance behaviors,” not just code paths

Examples:

* loader warns on shape drift
* loader hard-fails on missing files
* report contains “No probabilistic scoring”

That’s the right unit-test philosophy for this repo.

### ✅ `_segment_rollup` and `domain_status` now have a contract

This is key: you’re asserting `domain_status` exists in `finance_trend` in `test_compute_customer_trends_one_customer()`:

```python
assert "domain_status" in trends[0]["finance_trend"]
```

That means the segment view concern is resolved *as long as* `compute_customer_trends()` really sets it in all cases. Great—tests are enforcing the invariant.

---

## Two issues to fix (they’ll bite you later)

### 1) `test_load_cbo_data_success` likely fails with your current loader

Your loader resolves paths like:

```python
project_root = _resolve_project_root()
data_dir = project_root / config.data_dir
```

But in the test you set:

```python
config = CBOv2Config(data_dir=str(tmp_path))
```

If `data_dir` is an **absolute path** (tmp_path usually is), then `project_root / abs_path` is fine because `Path` joins with the absolute path (it ignores the left side). ✅

However, there’s a different problem:

Your `CBOv2Config` uses default filenames like:

* `customers_file = "customers.json"`
* `finance_signals_file = "finance_signals.json"`
* etc.

Your test writes exactly those filenames ✅

So this test is fine *if* your dataclass allows `CBOv2Config(data_dir=str(tmp_path))` to override `data_dir` via constructor. It should.

✅ Verdict: probably okay.

**But:** if Cursor later changes `_resolve_project_root` depth or changes `data_dir` to always be relative, this becomes brittle. I’d still keep it, but I’d add one more test to explicitly validate absolute-path support (see below).

---

### 2) `test_classify_cross_business_patterns_empty` has a weird invariant

You assert:

```python
assert out["portfolio_rollup"]["customers_total"] == 1  # max(0,1)
```

This matches your earlier pattern of `total = max(len(customer_trends), 1)` to avoid divide-by-zero.

But “customers_total = 1” when there are zero customers is semantically odd. It can confuse downstream reporting.

**Recommendation:** keep the divide-by-zero protection internally, but store:

* `customers_total = len(customer_trends)` (0)
* `pct_negative_alignment = 0.0` (guarded)
* `pct_positive_alignment = 0.0` (guarded)

Then your tests become more intuitive and your report reads more honestly.

If you keep the current logic (which is acceptable for MVP), fine—your test correctly locks it in. Just know it’s a “math hack” that leaks into semantics.

---

## Small test-quality improvements (low effort, high payoff)

### A) Add a test that validation warnings surface for empty domains

This is a core trust feature you added in v2.

Example:

* customers present
* finance empty
* expect warning: `"finance_signals is empty"`

### B) Add a test that “so-what” line appears only when stress threshold crossed

You introduced `portfolio_stress_threshold` behavior. Lock it in.

* pct_negative_alignment = 0.30
* threshold = 0.25
* expect “Portfolio stress increasing.”

And a second test where pct=0.10 and threshold=0.25 and it must NOT appear.

### C) Add a test for segment view correctness under insufficient_data

Since `_segment_rollup` depends on `domain_status`, add:

* one customer with finance/retail “insufficient_data”, healthcare “neutral”
* expected segment = “1-domain”

This protects your segmentation from future drift.

### D) Add test that drill-down doesn’t show empty parens

If no domains match, `_domain_drill_down` returns `""`. Ensure the report line doesn’t include `()`.

---

## One structural improvement to the test file

### Use `monkeypatch` to avoid touching sys.path

Right now you’re doing:

```python
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
```

It works, but it’s a little “test runner dependent.”

If your repo is a proper package, better to run from repo root and rely on standard imports. If not, keep it.

But since this is a template repo, I get why you want tests to be frictionless.

---

## The one change I’d make immediately (most important)

Add a test that asserts `compute_customer_trends` sets **domain_status for all three domains** for a customer, including when signals are missing.

Because segment view depends on it.

Example:

* customer exists
* healthcare_signals missing for customer
* assert healthcare_trend["domain_status"] == "insufficient_data"
* and segment rollup counts domains correctly

This will permanently prevent the “everything looks 3-domain” failure mode.

---

## Bottom line

This is a strong MVP test suite. You’re testing the right things.

If you implement just 3 additional tests, I’d do:

1. validation warning for empty domains
2. stress “so-what” line on/off
3. segment view with insufficient_data

That locks in the entire “executive trust” surface.




In [ ]:
"""
Unit tests for CBOv2 orchestrator utilities.

Test data loading, validation, analysis (trends, patterns), and report helpers.
Use minimal or mock data; no graph. Run from project root with: pytest test_CBOv2_utilities.py -v --tb=short
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

# Project root for imports
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pytest
from config import CBOv2Config, CBOv2State
from agents.CBOv2.orchestrator.utilities import data_loading
from agents.CBOv2.orchestrator.utilities.analysis import compute_customer_trends, classify_cross_business_patterns
from agents.CBOv2.orchestrator.utilities.reporting import (
    build_markdown_report,
    _build_verdict_line,
    _build_recommendation,
    _domain_drill_down,
    _segment_rollup,
)


# ----- Data loading -----

def test_validate_collection_empty():
    warnings = data_loading._validate_collection("customers", [], ["customer_id"])
    assert warnings == []


def test_validate_collection_missing_key():
    warnings = data_loading._validate_collection(
        "customers", [{"name": "A"}, {"customer_id": "C1"}], ["customer_id"]
    )
    assert len(warnings) == 1
    assert "missing required key 'customer_id'" in warnings[0]


def test_normalize_to_list_none():
    w: list = []
    out = data_loading._normalize_to_list(None, "x", w)
    assert out == []
    assert len(w) == 0


def test_normalize_to_list_list():
    w: list = []
    out = data_loading._normalize_to_list([{"a": 1}], "x", w)
    assert out == [{"a": 1}]
    assert len(w) == 0


def test_normalize_to_list_dict_appends_warning():
    w: list = []
    out = data_loading._normalize_to_list({"data": [{"id": 1}]}, "customers", w)
    assert out == [{"id": 1}]
    assert len(w) == 1
    assert "expected list, got dict" in w[0]


def test_load_cbo_data_success(tmp_path):
    (tmp_path / "customers.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "finance_signals.json").write_text(json.dumps([{"customer_id": "C1", "balance_short": 100, "balance_long": 90}]))
    (tmp_path / "retail_signals.json").write_text(json.dumps([{"customer_id": "C1", "spend_short": 50, "spend_long": 40}]))
    (tmp_path / "healthcare_signals.json").write_text(json.dumps([{"customer_id": "C1", "utilization_short": 2, "utilization_long": 1, "gaps_short": 5, "gaps_long": 10}]))
    config = CBOv2Config(data_dir=str(tmp_path))
    data, counts, warnings = data_loading.load_cbo_data(config)
    assert data["customers"] == [{"customer_id": "C1"}]
    assert counts["customers_count"] == 1
    assert counts["finance_signals_count"] == 1
    assert counts["retail_signals_count"] == 1
    assert counts["healthcare_signals_count"] == 1
    assert isinstance(warnings, list)


def test_load_cbo_data_missing_file(tmp_path):
    (tmp_path / "customers.json").write_text("[]")
    config = CBOv2Config(data_dir=str(tmp_path))
    with pytest.raises(FileNotFoundError, match="Expected JSON file not found"):
        data_loading.load_cbo_data(config)


def test_update_state_with_loaded_data():
    state: CBOv2State = {"errors": []}
    data = {"customers": [{"customer_id": "C1"}], "finance_signals": [], "retail_signals": [], "healthcare_signals": []}
    counts = {"customers_count": 1, "finance_signals_count": 0, "retail_signals_count": 0, "healthcare_signals_count": 0}
    warnings: list = []
    new_state = data_loading.update_state_with_loaded_data(state, data, counts, warnings)
    assert new_state["customers"] == [{"customer_id": "C1"}]
    assert new_state["portfolio_rollup"]["customers_count"] == 1
    assert new_state["validation_warnings"] == []


# ----- Analysis -----

def test_compute_customer_trends_empty():
    config = CBOv2Config()
    state: CBOv2State = {"customers": [], "finance_signals": [], "retail_signals": [], "healthcare_signals": []}
    out = compute_customer_trends(state, config)
    assert out["customer_trends"] == []


def test_compute_customer_trends_one_customer():
    config = CBOv2Config()
    state: CBOv2State = {
        "customers": [{"customer_id": "C1"}],
        "finance_signals": [{"customer_id": "C1", "balance_short": 110, "balance_long": 100, "missed_payments_short": 0, "missed_payments_long": 0}],
        "retail_signals": [{"customer_id": "C1", "spend_short": 120, "spend_long": 100, "frequency_short": 3, "frequency_long": 2}],
        "healthcare_signals": [{"customer_id": "C1", "utilization_short": 3, "utilization_long": 1, "gaps_short": 5, "gaps_long": 20}],
    }
    out = compute_customer_trends(state, config)
    trends = out["customer_trends"]
    assert len(trends) == 1
    assert trends[0]["customer_id"] == "C1"
    assert "finance_trend" in trends[0]
    assert "domain_status" in trends[0]["finance_trend"]
    assert trends[0]["domains_positive"] >= 0
    assert trends[0]["domains_negative"] >= 0


def test_classify_cross_business_patterns_empty():
    config = CBOv2Config()
    state: CBOv2State = {"customer_trends": [], "portfolio_rollup": {}}
    out = classify_cross_business_patterns(state, config)
    assert out["cross_business_patterns"] == []
    assert out["portfolio_rollup"]["customers_total"] == 1  # max(0,1)
    assert out["portfolio_rollup"]["customers_with_negative_alignment"] == 0
    assert out["portfolio_rollup"]["customers_with_positive_alignment"] == 0


# ----- Report helpers -----

def test_build_verdict_line_no_signals():
    line = _build_verdict_line({"customers_with_negative_alignment": 0, "customers_with_positive_alignment": 0}, [])
    assert "No cross-business risk or expansion signals" in line


def test_build_verdict_line_one_growth():
    line = _build_verdict_line({"customers_with_negative_alignment": 0, "customers_with_positive_alignment": 1}, [])
    assert "1 expansion opportunity" in line


def test_build_recommendation_no_lists():
    line = _build_recommendation([], [], [])
    assert "No immediate action" in line


def test_build_recommendation_top_growth():
    line = _build_recommendation([], [{"customer_id": "C003"}], [])
    assert "C003" in line
    assert "expansion or cross-sell" in line


def test_domain_drill_down_growth():
    rec = {
        "finance_trend": {"is_positive": True},
        "retail_trend": {"is_positive": False},
        "healthcare_trend": {"is_positive": True},
    }
    s = _domain_drill_down(rec, "growth")
    assert "Finance" in s and "Healthcare" in s
    assert "Retail" not in s or "Retail ✓" not in s


def test_segment_rollup_empty():
    assert _segment_rollup([]) == []


def test_segment_rollup_three_domain():
    trends = [
        {
            "finance_trend": {"domain_status": "neutral"},
            "retail_trend": {"domain_status": "neutral"},
            "healthcare_trend": {"domain_status": "neutral"},
            "domains_negative": 0,
            "domains_positive": 2,  # >= 2 counts as "in growth" for segment
        },
    ]
    rows = _segment_rollup(trends)
    assert len(rows) == 1
    assert rows[0][0] == "3-domain"
    assert rows[0][1] == 1
    assert rows[0][3] == 1  # in growth


def test_build_markdown_report_minimal_state():
    config = CBOv2Config()
    state: CBOv2State = {
        "portfolio_rollup": {"customers_total": 0, "customers_with_negative_alignment": 0, "customers_with_positive_alignment": 0},
        "executive_triggers": [],
        "top_risk_customers": [],
        "top_growth_customers": [],
        "customer_trends": [],
        "run_id": None,
        "config_snapshot": {},
        "validation_warnings": [],
    }
    report = build_markdown_report(state, config)
    assert "# Cross-Business Customer Intelligence Orchestrator (CBOv2)" in report
    assert "Verdict:" in report
    assert "Executive Summary" in report
    assert "Run Metadata" in report
    assert "Top Cross-Business Risk" in report
    assert "Top Cross-Business Growth" in report
    assert "No probabilistic scoring" in report
